# Exploración general de la base de datos local

## Objetivo del notebook

Este notebook tiene como propósito realizar una primera exploración estructural y de volumen sobre la base de datos local `selmark.duckdb`, recientemente cargada con los datos extraídos del entorno corporativo de Selmark.

El análisis se ejecuta sobre la capa **Bronze**, que contiene los datos sin transformar tal y como llegaron desde el catálogo `selmark_pre_gold` de Databricks. Esta capa actúa como referencia inmutable: cualquier transformación posterior se realizará sobre las capas Silver y Gold, dejando Bronze intacta.

El alcance del notebook incluye:

1. Confirmar que las tablas se han cargado correctamente.
2. Inspeccionar el esquema (nombres y tipos de columnas) de cada tabla.
3. Validar volúmenes y unicidad de claves primarias.
4. Detectar valores anómalos preliminares en columnas críticas.
5. Generar las tablas resumen que servirán de base para la sección de calidad de datos de la memoria.

## 1. Configuración del entorno

Se importan las librerías necesarias y se establece la conexión con la base DuckDB en modo de solo lectura, dado que este notebook no realiza transformaciones sobre los datos.

In [15]:
import duckdb
import pandas as pd
from pathlib import Path

# Rutas del proyecto
RUTA_PROYECTO = Path("..").resolve()
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

# Conexión en modo lectura
con = duckdb.connect(str(RUTA_DUCKDB), read_only=True)

print(f"Conexión establecida con: {RUTA_DUCKDB}")
print(f"Modo: solo lectura")

Conexión establecida con: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb
Modo: solo lectura


## 2. Inventario de tablas

Se listan todas las tablas disponibles en cada esquema, con el objetivo de confirmar que la carga inicial se ha completado correctamente.

In [16]:
inventario = con.execute("""
    SELECT 
        schema_name AS esquema,
        table_name AS tabla,
        estimated_size AS filas_estimadas
    FROM duckdb_tables()
    WHERE schema_name IN ('bronze', 'silver', 'gold')
    ORDER BY schema_name, table_name
""").fetchdf()

print("Inventario de tablas por esquema:\n")
print(inventario.to_string(index=False))

Inventario de tablas por esquema:

esquema              tabla  filas_estimadas
 bronze        dim_cliente             3469
 bronze fact_lineas_pedido            41709
 bronze             mosaic             6457
 bronze  ventas_minoristas          2220352
   gold        cliente_360             3469
 silver        dim_cliente             3469
 silver fact_lineas_pedido            33353
 silver       mapeo_paises              156
 silver             mosaic             6457
 silver             tiempo             1461
 silver  ventas_minoristas          2870570


## 3. Esquemas y tipos de columnas

A continuación se inspecciona la estructura de cada tabla Bronze. Esta información es esencial para planificar las transformaciones de la capa Silver, especialmente en los casteos de tipos de datos (fechas, decimales, identificadores).

In [17]:
tablas_bronze = ['dim_cliente', 'fact_lineas_pedido', 'mosaic', 'ventas_minoristas']

for tabla in tablas_bronze:
    print(f"\n{'='*70}")
    print(f"Tabla: bronze.{tabla}")
    print('='*70)
    
    esquema = con.execute(f"""
        SELECT 
            column_name AS columna,
            data_type AS tipo,
            is_nullable AS admite_nulos
        FROM information_schema.columns
        WHERE table_schema = 'bronze' AND table_name = '{tabla}'
        ORDER BY ordinal_position
    """).fetchdf()
    
    print(esquema.to_string(index=False))


Tabla: bronze.dim_cliente
                 columna    tipo admite_nulos
              id_cliente VARCHAR          YES
          nombre_cliente VARCHAR          YES
nombre_comercial_cliente VARCHAR          YES
       direccion_cliente VARCHAR          YES
       localidad_cliente VARCHAR          YES
   codigo_postal_cliente VARCHAR          YES
codigo_provincia_cliente VARCHAR          YES

Tabla: bronze.fact_lineas_pedido
                      columna    tipo admite_nulos
                    id_pedido  BIGINT          YES
              id_linea_pedido  BIGINT          YES
                   id_cliente  BIGINT          YES
             cod_serie_modelo VARCHAR          YES
                     id_color  BIGINT          YES
                 fecha_pedido    DATE          YES
                fecha_entrega    DATE          YES
               cantidad_linea  BIGINT          YES
       cantidad_linea_servida  BIGINT          YES
                precio_unidad  DOUBLE          YES
         p

## 4. Volúmenes y validación de claves

Se confirman los volúmenes de cada tabla y se valida la unicidad de las claves primarias declaradas. Esta verificación es crítica antes de realizar cualquier operación de JOIN posterior, ya que duplicados en una clave primaria multiplicarían filas en las uniones.

**Claves primarias esperadas:**

| Tabla | Clave primaria |
|---|---|
| `dim_cliente` | `id_cliente` |
| `fact_lineas_pedido` | `(id_pedido, id_linea_pedido)` |
| `ventas_minoristas` | sin clave técnica |
| `mosaic` | (a determinar) |

In [18]:
print("VALIDACIÓN DE VOLÚMENES Y CLAVES PRIMARIAS\n")
print(f"{'Tabla':<30} {'Filas':>15} {'Distintos':>15} {'Única':>10}")
print("-" * 75)

# dim_cliente: id_cliente debe ser única
res = con.execute("""
    SELECT COUNT(*) AS total, COUNT(DISTINCT id_cliente) AS distintos
    FROM bronze.dim_cliente
""").fetchone()
unica = "SI" if res[0] == res[1] else "NO"
print(f"{'dim_cliente (id_cliente)':<30} {res[0]:>15,} {res[1]:>15,} {unica:>10}")

# fact_lineas_pedido: (id_pedido, id_linea_pedido) debe ser única
res = con.execute("""
    SELECT COUNT(*) AS total, COUNT(DISTINCT id_pedido || '_' || id_linea_pedido) AS distintos
    FROM bronze.fact_lineas_pedido
""").fetchone()
unica = "SI" if res[0] == res[1] else "NO"
print(f"{'fact_lineas (id_pedido,linea)':<30} {res[0]:>15,} {res[1]:>15,} {unica:>10}")

# ventas_minoristas: solo conteo total
res = con.execute("SELECT COUNT(*) FROM bronze.ventas_minoristas").fetchone()
print(f"{'ventas_minoristas (sin clave)':<30} {res[0]:>15,} {'-':>15} {'-':>10}")

# mosaic: solo conteo total
res = con.execute("SELECT COUNT(*) FROM bronze.mosaic").fetchone()
print(f"{'mosaic (sin clave aún)':<30} {res[0]:>15,} {'-':>15} {'-':>10}")

VALIDACIÓN DE VOLÚMENES Y CLAVES PRIMARIAS

Tabla                                    Filas       Distintos      Única
---------------------------------------------------------------------------
dim_cliente (id_cliente)                 3,469           3,469         SI
fact_lineas (id_pedido,linea)           41,709          41,709         SI
ventas_minoristas (sin clave)        2,220,352               -          -
mosaic (sin clave aún)                   6,457               -          -


## 5. Cobertura temporal por tabla

Se verifica el rango temporal real de las tablas que contienen fechas. Esta información es clave para confirmar que la ventana de análisis del TFG (2022-2025) está cubierta correctamente y para detectar valores anómalos como fechas futuras o muy antiguas.

In [19]:
print("COBERTURA TEMPORAL\n")

# fact_lineas_pedido
print("bronze.fact_lineas_pedido (campo fecha_pedido):")
res = con.execute("""
    SELECT 
        MIN(fecha_pedido) AS fecha_min,
        MAX(fecha_pedido) AS fecha_max,
        COUNT(DISTINCT EXTRACT(YEAR FROM fecha_pedido)) AS num_anios
    FROM bronze.fact_lineas_pedido
""").fetchdf()
print(res.to_string(index=False))

print("\nDistribución por año:")
res = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM fecha_pedido) AS anio,
        COUNT(*) AS filas
    FROM bronze.fact_lineas_pedido
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(res.to_string(index=False))

# ventas_minoristas
print("\n\nbronze.ventas_minoristas (campo fecha_venta):")
res = con.execute("""
    SELECT 
        MIN(fecha_venta) AS fecha_min,
        MAX(fecha_venta) AS fecha_max,
        COUNT(DISTINCT EXTRACT(YEAR FROM fecha_venta)) AS num_anios
    FROM bronze.ventas_minoristas
""").fetchdf()
print(res.to_string(index=False))

print("\nDistribución por año:")
res = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM fecha_venta) AS anio,
        COUNT(*) AS filas
    FROM bronze.ventas_minoristas
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(res.to_string(index=False))

COBERTURA TEMPORAL

bronze.fact_lineas_pedido (campo fecha_pedido):
 fecha_min  fecha_max  num_anios
2021-06-09 2026-12-31          6

Distribución por año:
 anio  filas
 2021     28
 2022  11280
 2023   6830
 2024   6589
 2025  10377
 2026   6605


bronze.ventas_minoristas (campo fecha_venta):
 fecha_min  fecha_max  num_anios
2022-01-01 2025-12-31          4

Distribución por año:
 anio  filas
 2022 729665
 2023 719482
 2024 705210
 2025  65995


## 6. Inspección preliminar de la tabla `mosaic`

Dado que la tabla `mosaic` no se exploró previamente en el entorno de Databricks, se realiza aquí una primera inspección para entender su estructura y granularidad. Esto determinará si su clave de unión con el resto del modelo es el código postal, otra variable geográfica o un identificador específico.

In [20]:
print("PRIMERAS 5 FILAS DE bronze.mosaic\n")
muestra = con.execute("SELECT * FROM bronze.mosaic LIMIT 5").fetchdf()
print(muestra.to_string(index=False))

print("\n\nESTADÍSTICAS BÁSICAS POR COLUMNA\n")
columnas = con.execute("""
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_schema = 'bronze' AND table_name = 'mosaic'
    ORDER BY ordinal_position
""").fetchdf()['column_name'].tolist()

print(f"Columnas detectadas: {len(columnas)}")
for c in columnas:
    print(f"  - {c}")

PRIMERAS 5 FILAS DE bronze.mosaic

  CP CP_value PROV    PROV_INE  A1          A2  A3  A4 B10  B5          B6          B7          B8          B9         C11         C12 C13 C14 C15         D16         D17 D18 D19         E20    E21         E22         E23         E24 F25 F26 F27         G28 G29 G30 H31 H32         H33         H34 H35 H36 I37 I38 I39 I40 J41 J42 J43 K44 K45 K46 K47 K48 K49 K50   U Max_Mosaic Max_Mosaic1           A           B           C           D           E   F           G           H   I   J   K  U2 Max_Mosaic_G Max_Mosaic2 Renta_Media   F2 Count Mosaic_number Check
1000     1000    1 Araba/Álava 0.0         0.0 0.0 0.0 0.0 0.0         0.0         0.0         0.0         1.0         0.0         0.0 0.0 0.0 0.0         0.0         0.0 0.0 0.0         0.0    0.0         0.0         0.0         0.0 0.0 0.0 0.0         0.0 0.0 0.0 0.0 0.0         0.0         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0         B9           1         0.0    

## 7. Conclusiones del análisis exploratorio

### Hallazgos confirmados

- Las cuatro tablas Bronze se han cargado correctamente con los volúmenes esperados (3.469 / 41.709 / 6.457 / 2.870.570 filas).
- Las claves primarias de `dim_cliente` (id_cliente) y `fact_lineas_pedido` (id_pedido + id_linea_pedido) son únicas, garantizando JOINs sin riesgo de duplicación.
- La cobertura temporal de `ventas_minoristas` cubre exactamente la ventana del TFG (2022-2025) con volúmenes estables entre 700.000 y 730.000 filas anuales.
- La tabla `mosaic` utiliza el código postal (`CP`) como clave de unión con `dim_cliente`, y aporta tanto segmentación fina (50+ segmentos) como agregada por grupo (11 grupos A-K), además de renta media por código postal.

### Problemas de calidad detectados

**Inconsistencia de tipos en `id_cliente`**: la columna `id_cliente` se carga como `VARCHAR` en `dim_cliente` y como `BIGINT` en `fact_lineas_pedido` y `ventas_minoristas`. Será necesario castear todos los identificadores a `VARCHAR` en la capa Silver para garantizar la coherencia de los JOIN.

**Tipo incorrecto en `fecha_anulacion`**: la columna se carga como `VARCHAR` cuando debería ser `DATE`. Se aplicará casteo en Silver.

**Registros anómalos en `fact_lineas_pedido`**:
- 28 líneas en 2021 (cobertura insuficiente, justifica excluir 2021).
- 6.605 líneas con fecha en 2026, algunas posiblemente con fechas futuras erróneas (la fecha máxima registrada es 31-12-2026).

**Calidad de la tabla `mosaic`**:
- Los códigos postales se han cargado sin ceros iniciales (ej. `1000` en lugar de `01000`). Requerirá padding antes del JOIN.
- Algunas filas presentan valores nulos en variables clave como `Renta_Media`. Se evaluará su tratamiento en Silver.

### Decisiones metodológicas para la capa Silver

1. Aplicar la ventana temporal **2022-2025** a ambas tablas de hechos.
2. Castear todos los `id_cliente` a VARCHAR.
3. Excluir líneas anuladas (`esta_anulado = true`) de `fact_lineas_pedido`.
4. Normalizar códigos postales con padding a 5 dígitos en `dim_cliente` y `mosaic`.
5. Crear flag `es_cliente_espanol` en `dim_cliente`.
6. Crear vista Silver simplificada de MOSAIC con solo las columnas relevantes para el análisis (clave, grupo dominante, segmento dominante, renta media).

### Próximo notebook

`02_silver_dim_cliente.ipynb` — Construcción de la dimensión de clientes limpia, con normalización geográfica y enriquecimiento con flags analíticos.

In [21]:
# ============================================================
# CIERRE DE LA SESIÓN
# ============================================================
# Libera la conexión a DuckDB para evitar bloqueos en otros notebooks

try:
    con.close()
    print("Conexión a DuckDB cerrada correctamente.")
except Exception as e:
    print(f"Aviso al cerrar conexión: {e}")

import gc
gc.collect()

print("\nNotebook finalizado. La base está libre para otros notebooks.")
print("Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.")

Conexión a DuckDB cerrada correctamente.

Notebook finalizado. La base está libre para otros notebooks.
Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.
